In [3]:
import polars as pl

from data_engineering.openairframes.read import scan_latest_registration_icao_for_join

EUROCONTROL_PATH = "/Volumes/T2-SSD/planequery/data/raw/eurocontrol/202406/intermediate/flights_20240601_20240630_time_columns.parquet"

df_eurocontrol = pl.read_parquet(EUROCONTROL_PATH)

In [4]:
COLUMNS = ["AC Registration", "ADEP", "ADES", "ACTUAL OFF BLOCK TIME", "ACTUAL ARRIVAL TIME"]
REGISTRATION_JOIN_KEY = "registration_join_key"

df_openairframes = (
    scan_latest_registration_icao_for_join(real_icao_column="icao")
    .with_columns(
        pl.col("registration")
        .str.replace_all(r"[^A-Z0-9]", "")
        .alias(REGISTRATION_JOIN_KEY)
    )
    .filter(pl.col(REGISTRATION_JOIN_KEY) != "")
    .drop("registration")
    .unique(subset=[REGISTRATION_JOIN_KEY], keep="last", maintain_order=False)
)

df = (
    df_eurocontrol.select(COLUMNS)
    .with_columns(
        pl.col("AC Registration")
        .str.strip_chars()
        .str.to_uppercase()
        .str.replace_all(r"[^A-Z0-9]", "")
        .alias(REGISTRATION_JOIN_KEY)
    )
    .lazy()
    .join(df_openairframes, on=REGISTRATION_JOIN_KEY, how="left")
    .drop(REGISTRATION_JOIN_KEY)
    .select(["AC Registration", "icao", "ADEP", "ADES", "ACTUAL OFF BLOCK TIME", "ACTUAL ARRIVAL TIME"])
    .collect()
)
df

AC Registration,icao,ADEP,ADES,ACTUAL OFF BLOCK TIME,ACTUAL ARRIVAL TIME
str,str,str,str,datetime[μs],datetime[μs]
"""N588TN""","""a79556""","""KPBI""","""EGLF""",2024-06-01 00:43:00,2024-06-01 08:53:00
"""N44WG""","""a54a3f""","""KFLL""","""LIMJ""",2024-06-01 00:09:00,2024-06-01 09:28:00
"""N686UA""","""a91b44""","""KEWR""","""EGLL""",2024-06-01 00:55:00,2024-06-01 07:50:46
"""N24973""","""a254d7""","""KLAX""","""EGLL""",2024-06-01 00:45:00,2024-06-01 10:38:57
"""9HVJO""","""4d20f7""","""KOPF""","""EGLF""",2024-06-01 00:47:00,2024-06-01 08:59:00
…,…,…,…,…,…
"""PKLEW""","""8a09df""","""GMMN""","""OEJN""",2024-07-01 00:21:42,2024-07-01 06:05:38
"""N761AJ""","""aa4548""","""KDFW""","""EIDW""",2024-07-01 01:18:00,2024-07-01 09:04:15
"""YIARI""","""728649""","""LTFM""","""ORBI""",2024-07-01 00:07:00,2024-07-01 02:51:17


In [5]:
df.select(
    pl.len().alias("rows"),
    pl.col("icao").is_not_null().sum().alias("rows_with_icao"),
    pl.col("icao").is_null().sum().alias("rows_missing_icao"),
)

rows,rows_with_icao,rows_missing_icao
u32,u32,u32
935934,932865,3069


In [6]:
df

AC Registration,icao,ADEP,ADES,ACTUAL OFF BLOCK TIME,ACTUAL ARRIVAL TIME
str,str,str,str,datetime[μs],datetime[μs]
"""N588TN""","""a79556""","""KPBI""","""EGLF""",2024-06-01 00:43:00,2024-06-01 08:53:00
"""N44WG""","""a54a3f""","""KFLL""","""LIMJ""",2024-06-01 00:09:00,2024-06-01 09:28:00
"""N686UA""","""a91b44""","""KEWR""","""EGLL""",2024-06-01 00:55:00,2024-06-01 07:50:46
"""N24973""","""a254d7""","""KLAX""","""EGLL""",2024-06-01 00:45:00,2024-06-01 10:38:57
"""9HVJO""","""4d20f7""","""KOPF""","""EGLF""",2024-06-01 00:47:00,2024-06-01 08:59:00
…,…,…,…,…,…
"""PKLEW""","""8a09df""","""GMMN""","""OEJN""",2024-07-01 00:21:42,2024-07-01 06:05:38
"""N761AJ""","""aa4548""","""KDFW""","""EIDW""",2024-07-01 01:18:00,2024-07-01 09:04:15
"""YIARI""","""728649""","""LTFM""","""ORBI""",2024-07-01 00:07:00,2024-07-01 02:51:17
